# Worksheet 8: Decision Tree, Ensemble Methods and Hyperparameter Tuning

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, mean_squared_error, r2_score
from scipy.stats import randint, uniform
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load Wine Dataset
wine = load_wine()
df = pd.DataFrame(data=wine.data, columns=wine.feature_names)
df['target'] = wine.target
print(f"Shape: {df.shape}, Classes: {wine.target_names}")
df.head()

Shape: (178, 14), Classes: ['class_0' 'class_1' 'class_2']


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [ ]:
# Train-Test Split
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

Train: 142, Test: 36


## Task 1: Classification - Decision Tree vs Random Forest

In [ ]:
# Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)
dt_predictions = dt_classifier.predict(X_test)
dt_f1 = f1_score(y_test, dt_predictions, average='weighted')
print(f"Decision Tree F1 Score: {dt_f1:.4f}")

Decision Tree F1 Score: 0.9450


In [ ]:
# Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)
rf_predictions = rf_classifier.predict(X_test)
rf_f1 = f1_score(y_test, rf_predictions, average='weighted')
print(f"Random Forest F1 Score: {rf_f1:.4f}")

Random Forest F1 Score: 1.0000


In [ ]:
# Task 1 Comparison
print(f"Decision Tree F1: {dt_f1:.4f}")
print(f"Random Forest F1: {rf_f1:.4f}")
print(f"Winner: {'Random Forest' if rf_f1 > dt_f1 else 'Decision Tree'}")

Decision Tree F1: 0.9450
Random Forest F1: 1.0000
Winner: Random Forest


## Task 2: GridSearchCV - Tune 3 Hyperparameters

In [ ]:
# GridSearchCV with 3 hyperparameters: n_estimators, max_depth, min_samples_split
param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10]
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)
print(f"Best Params: {grid_search.best_params_}")
print(f"Best CV F1: {grid_search.best_score_:.4f}")

Best Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}
Best CV F1: 0.9860


In [ ]:
# Tuned Model Evaluation
best_rf = grid_search.best_estimator_
best_rf_predictions = best_rf.predict(X_test)
best_rf_f1 = f1_score(y_test, best_rf_predictions, average='weighted')
print(f"Tuned RF F1 Score: {best_rf_f1:.4f}")

Tuned RF F1 Score: 1.0000


## Task 3: Regression with RandomizedSearchCV

In [ ]:
# Regression Data - Predict alcohol content
df_reg = pd.DataFrame(data=wine.data, columns=wine.feature_names)
X_reg = df_reg.drop('alcohol', axis=1)
y_reg = df_reg['alcohol']
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

In [ ]:
# Decision Tree Regressor
dt_regressor = DecisionTreeRegressor(random_state=42)
dt_regressor.fit(X_train_reg, y_train_reg)
dt_reg_r2 = r2_score(y_test_reg, dt_regressor.predict(X_test_reg))
print(f"Decision Tree R²: {dt_reg_r2:.4f}")

Decision Tree R²: 0.4775


In [ ]:
# Random Forest Regressor
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_reg, y_train_reg)
rf_reg_r2 = r2_score(y_test_reg, rf_regressor.predict(X_test_reg))
print(f"Random Forest R²: {rf_reg_r2:.4f}")

Random Forest R²: 0.7416


In [ ]:
# RandomizedSearchCV with 3 parameters: n_estimators, max_depth, min_samples_leaf
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [None, 5, 10, 15, 20, 25, 30],
    'min_samples_leaf': randint(1, 10)
}
random_search = RandomizedSearchCV(RandomForestRegressor(random_state=42), param_dist, n_iter=50, cv=5, scoring='r2', n_jobs=-1, random_state=42)
random_search.fit(X_train_reg, y_train_reg)
print(f"Best Params: {random_search.best_params_}")
print(f"Best CV R²: {random_search.best_score_:.4f}")

Best Params: {'max_depth': 30, 'min_samples_leaf': 9, 'n_estimators': 256}
Best CV R²: 0.5181


In [ ]:
# Tuned Regressor Evaluation
best_rf_reg = random_search.best_estimator_
best_rf_reg_r2 = r2_score(y_test_reg, best_rf_reg.predict(X_test_reg))
print(f"Tuned RF Regressor R²: {best_rf_reg_r2:.4f}")

Tuned RF Regressor R²: 0.7564
